In [ ]:
!sudo apt-get install zstd
# Instalar Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Iniciar el servidor de Ollama en segundo plano
import subprocess
import threading
import time

def run_ollama():
    subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama)
thread.start()
time.sleep(5) # Dar tiempo a que el servidor inicie
print("Servidor de Ollama iniciado.")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (548 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently i

In [ ]:
!ollama pull glm-5.1:cloud #magistral:24b #ministral-3:14b #qwen3:1.7b #qwen2.5:3b #glm4:9b #mistral:7b llama3.2:3b

In [ ]:
# %% [markdown]
# # PoliticHeadlinES 2026 — Baseline con LLMs Multi-Plataforma
#
# Este notebook implementa un baseline avanzado utilizando:
# - **Task 1**: Modelos de lenguaje para ranking semántico
# - **Task 2**: Fusión multimodal con CLIP + LLM
#
# ## Opciones de Modelos Disponibles:
# | Plataforma | Tipo | Configuración |
# |------------|------|---------------|
# | Hugging Face Local | Transformers | `METHOD_LLM = "hf_local"` |
# | Hugging Face Inference API | Cloud API | `METHOD_LLM = "hf_api"` |
# | Ollama Local | Local Server | `METHOD_LLM = "ollama_local"` |
# | Ollama Remoto | Cloud/Remote | `METHOD_LLM = "ollama_remote"` |
# | Sentence Transformers | Embeddings | `METHOD_LLM = "embedding"` |
# | NLI Cross-Encoder | Inferencia | `METHOD_LLM = "nli"` |

# %% [markdown]
# ## Instalación de Dependencias

# %%
!pip -q install pandas numpy scikit-learn torch torchvision transformers accelerate sentence-transformers pillow requests
!pip -q install huggingface_hub ollama python-dotenv

# %%
from __future__ import annotations
import hashlib, json, math, os, warnings, time, requests
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from PIL import Image
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

# Hugging Face
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    AutoModel, pipeline, CLIPProcessor, CLIPModel
)
from sentence_transformers import SentenceTransformer, util
from huggingface_hub import InferenceClient

# Ollama
try:
    import ollama
    OLLAMA_AVAILABLE = True
except ImportError:
    OLLAMA_AVAILABLE = False
    print("⚠️ Ollama no instalado. pip install ollama")

warnings.filterwarnings("ignore")
load_dotenv()  # Cargar variables de entorno desde .env

# %% [markdown]
# ## Configuración Global

# %%
# ========== CONFIGURACIÓN PRINCIPAL ==========
METHOD_LLM = "ollama_local"  # "hf_local" | "hf_api" | "ollama_local" | "ollama_remote" | "embedding" | "nli"
METHOD_MULTIMODAL = "clip_llm_fusion"  # "clip_llm_fusion" | "clip_only" | "llm_multimodal"

# ========== HUGGING FACE CONFIG ==========
HF_LOCAL_MODEL = "PlanTL-GOB-ES/roberta-base-bne"  # Modelo local para español
HF_API_MODEL = "mistralai/Mistral-7B-Instruct-v0.3" # "gpt2" bert-base-uncased facebook/bart-large-cnn # Modelo para Inference API
HF_API_TOKEN = "" #os.getenv("HF_TOKEN", "")  # Token desde .env o variables

# ========== OLLAMA CONFIG ==========
OLLAMA_MODEL =  "glm-5.1:cloud" # "ministral-3:14b" #"mistral:7b" #"llama3.2:3b"  # Modelo Ollama
OLLAMA_HOST_LOCAL = "http://localhost:11434"  # Ollama local
OLLAMA_HOST_REMOTE = "https://ollama.com/"  #os.getenv("OLLAMA_HOST", "http://localhost:11434")  # Ollama remoto (cloud)
OLLAMA_TOKEN = "" #os.getenv("OLLAMA_TOKEN", "")  # Si requiere autenticación

# ========== EMBEDDINGS & NLI ==========
EMBEDDING_MODEL = "sentence-transformers/paraphrasing-multilingual-mpnet-base-v2"
NLI_MODEL = "cross-encoder/nli-deberta-v3-base"

# ========== CLIP MULTIMODAL ==========
CLIP_MODEL = "openai/clip-vit-base-patch32"

# ========== PARÁMETROS ==========
TEST_SIZE = 0.2
SEED = 42
NDCG_K = 10
ALPHA = 0.9
W_TEXT = 0.85  # Peso para texto en fusión multimodal
W_IMAGE = 0.15  # Peso para imagen en fusión multimodal

# ========== RUTAS ==========
TRAIN_CSV = "train_public.csv"
DEV_CSV = "dev_public.csv"
OUTPUT_SUBMISSION = "results_llm.csv"
IMAGES_DIR = Path("images")

TITLE_COLS = [f"title_{i}" for i in range(1, 11)]
TOKENS_ALL = [f"t{i}" for i in range(1, 11)]
N_COLS = 10

# ========== DEVICE ==========
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✓ Device: {device}")
print(f"✓ Method LLM: {METHOD_LLM}")
print(f"✓ Method Multimodal: {METHOD_MULTIMODAL}")

# %% [markdown]
# ## Descarga y Carga de Datos

# %%
def validate_columns(df: pd.DataFrame) -> None:
    required = ["id", "article_body", "image_hash"] + TITLE_COLS
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Faltan columnas: {missing}")

# Descargar dataset si no existe
if not os.path.exists(TRAIN_CSV):
    DATASET_URL = "https://pln.inf.um.es/corpora/politicheadlines/2026/development_phase_dataset.zip"
    print("📥 Descargando dataset...")
    !wget -q $DATASET_URL -O development_phase_dataset.zip
    !unzip -q development_phase_dataset.zip
    print("✓ Dataset extraído")

df_train = pd.read_csv(TRAIN_CSV, dtype={"id": str})
df_test = pd.read_csv(DEV_CSV, dtype={"id": str})
validate_columns(df_train)
validate_columns(df_test)

print(f"✓ Train: {len(df_train)} | Test: {len(df_test)} registros")

# %% [markdown]
# ## Funciones Auxiliares

# %%
def tokens(x: Any) -> List[str]:
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    s = str(x).strip()
    return s.split() if s else []

def get_source_text(row: pd.Series) -> str:
    return str(row.get("article_body", "") or "").strip()

def get_titles(row: pd.Series) -> List[str]:
    return [str(row.get(c, "") or "").strip() for c in TITLE_COLS]

def find_image_path(images_dir: Path, image_hash: str) -> Optional[Path]:
    if not image_hash or (isinstance(image_hash, float) and np.isnan(image_hash)):
        return None
    h = str(image_hash).strip()
    if not h:
        return None
    for ext in [".jpg", ".jpeg", ".png", ".webp"]:
        p = images_dir / f"{h}{ext}"
        if p.exists():
            return p
    p = images_dir / h
    return p if p.exists() else None

def _minmax_01(x: np.ndarray) -> np.ndarray:
    x = x.astype(float)
    mn, mx = float(np.min(x)), float(np.max(x))
    if mx - mn < 1e-12:
        return np.zeros_like(x, dtype=float)
    return (x - mn) / (mx - mn)

# %% [markdown]
# ## Task 1: Ranking con LLMs (Text-only)

# %% [markdown]
# ### Opción 1: Hugging Face Local (Transformers)

# %%
class HF_Local_Ranker:
    """Ranking usando modelos locales de Hugging Face"""

    def __init__(self, model_name: str = HF_LOCAL_MODEL):
        print(f"🔍 Cargando modelo HF Local: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=1,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
            device_map="auto" if device == "cuda" else None
        ).eval()
        self.model_name = model_name

    def _score_pair(self, article: str, title: str) -> float:
        """Calcula score de relevancia entre artículo y titular"""
        inputs = self.tokenizer(
            article[:512], title,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding=True
        ).to(self.model.device)

        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits
            score = torch.sigmoid(logits).item()
        return score

    def rank_titles(self, article: str, titles: List[str]) -> List[int]:
        scores = []
        for title in titles:
            score = self._score_pair(article, title)
            scores.append(score)
        order = np.argsort(-np.array(scores))
        return order.tolist()

# %% [markdown]
# ### Opción 2: Hugging Face Inference API (Cloud)

# %%
class HF_API_Ranker:
    """Ranking usando Hugging Face Inference API (Cloud)"""

    def __init__(self, model_name: str = HF_API_MODEL, token: str = HF_API_TOKEN):
        print(f"☁️ Conectando a HF Inference API: {model_name}")
        if not token:
            print("⚠️ HF_TOKEN no configurado. Algunas APIs pueden tener rate limits.")
        self.client = InferenceClient(model=model_name, token=token if token else None)
        self.model_name = model_name

    def _score_with_prompt(self, article: str, title: str) -> float:
        """Usa prompting para obtener score de relevancia"""
        prompt = f"""Eres un evaluador de noticias políticas en español.
Tu tarea es evaluar qué tan bien este titular resume el artículo.

ARTÍCULO:
{article[:1000]}...

TITULAR:
{title}

Responde SOLO con un número del 0 al 10, donde:
- 0 = El titular no tiene relación con el artículo
- 10 = El titular resume perfectamente el artículo

Puntuación:"""

        try:
            response = self.client.text_generation(
                prompt,
                max_new_tokens=5,
                temperature=0.1,
                do_sample=False
            )
            # Extraer número
            import re
            numbers = re.findall(r'\d+\.?\d*', response)
            if numbers:
                score = float(numbers[0]) / 10.0
                return min(1.0, max(0.0, score))
            return 0.5
        except Exception as e:
            print(f"⚠️ Error en HF API: {e}")
            return 0.5

    def rank_titles(self, article: str, titles: List[str]) -> List[int]:
        scores = []
        for title in tqdm(titles, desc="HF API Scoring", leave=False):
            score = self._score_with_prompt(article, title)
            scores.append(score)
        order = np.argsort(-np.array(scores))
        return order.tolist()

# %% [markdown]
# ### Opción 3: Ollama Local

# %%
class Ollama_Local_Ranker:
    """Ranking usando Ollama en servidor local"""

    def __init__(self, model_name: str = OLLAMA_MODEL, host: str = OLLAMA_HOST_LOCAL):
        print(f"🦙 Conectando a Ollama Local: {model_name} @ {host}")
        self.model_name = model_name
        self.host = host
        self.client = ollama.Client(host=host)

        # Verificar conexión
        try:
            self.client.list()
            print("✓ Conexión a Ollama establecida")
        except Exception as e:
            print(f"⚠️ Error conectando a Ollama: {e}")

    def _score_with_prompt(self, article: str, title: str) -> float:
        prompt = f"""Eres un evaluador de noticias políticas en español.
Evalúa qué tan bien este titular resume el artículo.

ARTÍCULO:
{article[:1000]}...

TITULAR:
{title}

Responde SOLO con un número del 0 al 10.

Puntuación:"""

        try:
            response = self.client.generate(
                model=self.model_name,
                prompt=prompt,
                options={"temperature": 0.1, "num_predict": 5}
            )
            text = response['response'].strip()
            import re
            numbers = re.findall(r'\d+\.?\d*', text)
            if numbers:
                score = float(numbers[0]) / 10.0
                return min(1.0, max(0.0, score))
            return 0.5
        except Exception as e:
            print(f"⚠️ Error en Ollama: {e}")
            return 0.5

    def rank_titles(self, article: str, titles: List[str]) -> List[int]:
        scores = []
        for title in tqdm(titles, desc="Ollama Local Scoring", leave=False):
            score = self._score_with_prompt(article, title)
            scores.append(score)
        order = np.argsort(-np.array(scores))
        return order.tolist()

# %% [markdown]
# ### Opción 4: Ollama Remoto/Cloud

# %%
class Ollama_Remote_Ranker:
    """Ranking usando Ollama en servidor remoto (cloud)"""

    def __init__(self, model_name: str = OLLAMA_MODEL, host: str = OLLAMA_HOST_REMOTE, token: str = OLLAMA_TOKEN):
        print(f"☁️ Conectando a Ollama Remoto: {model_name} @ {host}")
        self.model_name = model_name
        self.host = host
        self.token = token
        self.client = ollama.Client(host=host)

        # Verificar conexión
        try:
            self.client.list()
            print("✓ Conexión a Ollama Remoto establecida")
        except Exception as e:
            print(f"⚠️ Error conectando a Ollama Remoto: {e}")

    def _score_with_prompt(self, article: str, title: str) -> float:
        prompt = f"""Eres un evaluador de noticias políticas en español.
Evalúa qué tan bien este titular resume el artículo.

ARTÍCULO:
{article[:1000]}...

TITULAR:
{title}

Responde SOLO con un número del 0 al 10.

Puntuación:"""

        try:
            response = self.client.generate(
                model=self.model_name,
                prompt=prompt,
                options={"temperature": 0.1, "num_predict": 5}
            )
            text = response['response'].strip()
            import re
            numbers = re.findall(r'\d+\.?\d*', text)
            if numbers:
                score = float(numbers[0]) / 10.0
                return min(1.0, max(0.0, score))
            return 0.5
        except Exception as e:
            print(f"⚠️ Error en Ollama Remoto: {e}")
            return 0.5

    def rank_titles(self, article: str, titles: List[str]) -> List[int]:
        scores = []
        for title in tqdm(titles, desc="Ollama Remote Scoring", leave=False):
            score = self._score_with_prompt(article, title)
            scores.append(score)
        order = np.argsort(-np.array(scores))
        return order.tolist()

# %% [markdown]
# ### Opción 5: Sentence Transformers (Embeddings)

# %%
class EmbeddingRanker:
    """Ranking basado en similitud de embeddings multilingües"""

    def __init__(self, model_name: str = EMBEDDING_MODEL):
        print(f"🔍 Cargando modelo de embeddings: {model_name}")
        self.model = SentenceTransformer(model_name, device=device)

    def rank_titles(self, article: str, titles: List[str]) -> List[int]:
        article_emb = self.model.encode([article], convert_to_tensor=True, show_progress_bar=False)
        title_embs = self.model.encode(titles, convert_to_tensor=True, show_progress_bar=False)
        sims = util.cos_sim(article_emb, title_embs)[0].cpu().numpy()
        order = np.argsort(-sims)
        return order.tolist()

# %% [markdown]
# ### Opción 6: NLI Cross-Encoder

# %%
class NLIRanker:
    """Ranking basado en inferencia natural (NLI)"""

    def __init__(self, model_name: str = NLI_MODEL):
        print(f"🔍 Cargando modelo NLI: {model_name}")
        self.classifier = pipeline(
            "text-classification",
            model=model_name,
            tokenizer=model_name,
            device=0 if device == "cuda" else -1,
            batch_size=8,
            truncation=True,
            max_length=512
        )

    def rank_titles(self, article: str, titles: List[str]) -> List[int]:
        scores = []
        for title in titles:
            result = self.classifier({"text": article, "text_pair": title})
            score = next((r['score'] for r in result if 'entail' in r['label'].lower()),
                        result[0]['score'] if result else 0.0)
            scores.append(score)
        order = np.argsort(-np.array(scores))
        return order.tolist()

# %% [markdown]
# ### Función Unificada para Task 1

# %%
def get_task1_ranker(method: str = METHOD_LLM):
    """Factory para obtener el ranker según configuración"""
    if method == "hf_local":
        return HF_Local_Ranker()
    elif method == "hf_api":
        return HF_API_Ranker()
    elif method == "ollama_local":
        return Ollama_Local_Ranker()
    elif method == "ollama_remote":
        return Ollama_Remote_Ranker()
    elif method == "embedding":
        return EmbeddingRanker()
    elif method == "nli":
        return NLIRanker()
    else:
        raise ValueError(f"Método LLM no reconocido: {method}")

def predict_task1_llm(df_train: pd.DataFrame, df_pred: pd.DataFrame) -> pd.Series:
    """Predice ranking para Task 1 usando el método LLM configurado"""
    ranker = get_task1_ranker(METHOD_LLM)

    preds = []
    for _, row in tqdm(df_pred.iterrows(), total=len(df_pred), desc="Task 1"):
        article = get_source_text(row)
        titles = get_titles(row)
        order = ranker.rank_titles(article, titles)
        ranked_tokens = [f"t{i+1}" for i in order]
        preds.append(" ".join(ranked_tokens))

    return pd.Series(preds, index=df_pred.index)

# %% [markdown]
# ## Task 2: Ranking Multimodal (Texto + Imagen)

# %% [markdown]
# ### Opción A: CLIP + LLM Fusion

# %%
class ClipLLMFusionRanker:
    """Fusión ponderada: CLIP (imagen) + LLM/Embeddings (texto)"""

    def __init__(self, clip_model_name: str = CLIP_MODEL,
                 text_ranker_method: str = METHOD_LLM,
                 w_text: float = W_TEXT, w_img: float = W_IMAGE):
        print(f"🔍 Cargando CLIP: {clip_model_name}")
        self.clip_model = CLIPModel.from_pretrained(clip_model_name).to(device)
        self.clip_processor = CLIPProcessor.from_pretrained(clip_model_name)

        print(f"🔍 Inicializando ranker de texto: {text_ranker_method}")
        self.text_ranker = get_task1_ranker(text_ranker_method)

        self.w_text = w_text
        self.w_img = w_img
        self.clip_model.eval()

    @torch.inference_mode()
    def _get_clip_scores(self, image_path: Path, titles: List[str]) -> np.ndarray:
        image = Image.open(image_path).convert("RGB")
        inputs = self.clip_processor(text=titles, images=image, return_tensors="pt", padding=True).to(device)
        outputs = self.clip_model(**inputs)
        return outputs.logits_per_image[0].detach().float().cpu().numpy()

    def rank_titles(self, article: str, titles: List[str], image_path: Optional[Path]) -> List[int]:
        # 1. Similitud texto-artículo (LLM/Embeddings)
        text_order = self.text_ranker.rank_titles(article, titles)
        text_scores = np.zeros(len(titles))
        for i, idx in enumerate(text_order):
            text_scores[idx] = len(titles) - i  # Score basado en ranking

        # 2. Similitud imagen-títulos (CLIP)
        if image_path and image_path.exists():
            img_scores = self._get_clip_scores(image_path, titles)
        else:
            img_scores = np.zeros_like(text_scores)

        # 3. Normalizar y fusionar
        text01 = _minmax_01(text_scores)
        img01 = _minmax_01(img_scores)
        scores = (self.w_text * text01) + (self.w_img * img01)

        order = np.argsort(-scores)
        return order.tolist()

# %% [markdown]
# ### Opción B: CLIP Only (Solo imagen)

# %%
class ClipOnlyRanker:
    """Ranking usando solo CLIP (imagen-título)"""

    def __init__(self, clip_model_name: str = CLIP_MODEL):
        print(f"🔍 Cargando CLIP: {clip_model_name}")
        self.clip_model = CLIPModel.from_pretrained(clip_model_name).to(device)
        self.clip_processor = CLIPProcessor.from_pretrained(clip_model_name)
        self.clip_model.eval()

    @torch.inference_mode()
    def rank_titles(self, article: str, titles: List[str], image_path: Optional[Path]) -> List[int]:
        if not image_path or not image_path.exists():
            # Fallback a orden identidad
            return list(range(len(titles)))

        image = Image.open(image_path).convert("RGB")
        inputs = self.clip_processor(text=titles, images=image, return_tensors="pt", padding=True).to(device)
        outputs = self.clip_model(**inputs)
        sims = outputs.logits_per_image[0].detach().float().cpu().numpy()
        order = np.argsort(-sims)
        return order.tolist()

# %% [markdown]
# ### Función Unificada para Task 2

# %%
def get_task2_ranker(method: str = METHOD_MULTIMODAL):
    """Factory para obtener el ranker multimodal según configuración"""
    if method == "clip_llm_fusion":
        return ClipLLMFusionRanker()
    elif method == "clip_only":
        return ClipOnlyRanker()
    else:
        print(f"⚠️ Método {method} no implementado. Usando clip_llm_fusion.")
        return ClipLLMFusionRanker()

def predict_task2_multimodal(df_train: pd.DataFrame, df_pred: pd.DataFrame, images_dir: Path) -> pd.Series:
    """Predice ranking para Task 2 usando enfoque multimodal"""
    ranker = get_task2_ranker(METHOD_MULTIMODAL)

    preds = []
    for _, row in tqdm(df_pred.iterrows(), total=len(df_pred), desc="Task 2"):
        article = get_source_text(row)
        titles = get_titles(row)
        img_path = find_image_path(images_dir, row.get("image_hash"))

        order = ranker.rank_titles(article, titles, img_path)
        ranked_tokens = [f"t{i+1}" for i in order]
        preds.append(" ".join(ranked_tokens))

    return pd.Series(preds, index=df_pred.index)

# %% [markdown]
# ## Ejecución y Generación de Submission

# %%
print(f"🚀 Ejecutando Task 1 con método: {METHOD_LLM}")
df_test["task_1"] = predict_task1_llm(df_train, df_test)

print(f"🚀 Ejecutando Task 2 con método: {METHOD_MULTIMODAL}")
df_test["task_2"] = predict_task2_multimodal(df_train, df_test, IMAGES_DIR)

# Guardar submission
submission = df_test[["id", "task_1", "task_2"]].copy()
submission.to_csv(OUTPUT_SUBMISSION, index=False)
print(f"✅ Submission guardado: {OUTPUT_SUBMISSION}")
display(submission.head())

# %% [markdown]
# ## Evaluación (si hay ground truth disponible)

# %%
def _parse_rank_list(x: Any) -> List[str]:
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    s = str(x).strip()
    if not s:
        return []
    if s.startswith("[") and s.endswith("]"):
        try:
            arr = json.loads(s)
            if isinstance(arr, list):
                return [str(t).strip() for t in arr if str(t).strip()]
        except:
            pass
    s = s.replace("\t", " ").replace("\n", " ").replace(";", " ")
    parts = [p.strip() for p in (s.split(",") if "," in s else s.split())]
    return [p for p in parts if p]

def _token_to_col(tok: Any) -> Optional[int]:
    if tok is None or (isinstance(tok, float) and pd.isna(tok)):
        return None
    s = str(tok).strip()
    if len(s) < 2 or s[0].lower() not in ("t", "d"):
        return None
    try:
        return int(s[1:])
    except:
        return None

def _unique_valid_cols(tokens: List[str], n_cols: int) -> List[int]:
    out, seen = [], set()
    for tok in tokens:
        n = _token_to_col(tok)
        if n and 1 <= n <= n_cols and n not in seen:
            seen.add(n)
            out.append(n)
    return out

def _ndcg(pred: List[int], ideal: List[int], k: int) -> float:
    if not ideal:
        return 0.0
    ideal_rank = {c: i for i, c in enumerate(ideal)}
    def gain(c):
        r = ideal_rank.get(c)
        return float(len(ideal) - r) if r is not None else 0.0

    dcg = sum(gain(c) / math.log2(i+2) for i, c in enumerate(pred[:k]))
    idcg = sum(gain(c) / math.log2(i+2) for i, c in enumerate(ideal[:k]))
    return max(0.0, min(1.0, dcg / idcg)) if idcg > 0 else 0.0

def pa_ndcg_score(pred_tokens: List[str], true_tokens: List[str], k: int = 10, alpha: float = 0.9) -> float:
    ideal = _unique_valid_cols(true_tokens, N_COLS)
    pred = _unique_valid_cols(pred_tokens, N_COLS)
    if not ideal or not pred or pred[0] != ideal[0]:
        return 0.0
    primary = ideal[0]
    aux = _ndcg([c for c in pred if c != primary], [c for c in ideal if c != primary], k)
    return alpha + (1 - alpha) * aux

def evaluate_submission(validation_csv: str, results_csv: str) -> Dict[str, float]:
    ref = pd.read_csv(validation_csv, dtype={"id": str})[["id", "y_true"]]
    sub = pd.read_csv(results_csv, dtype={"id": str})[["id", "task_1", "task_2"]]
    merged = ref.merge(sub, on="id", how="left")

    t1_scores = [pa_ndcg_score(_parse_rank_list(r["task_1"]), _parse_rank_list(r["y_true"]))
                 for _, r in merged.iterrows()]
    t2_scores = [pa_ndcg_score(_parse_rank_list(r["task_2"]), _parse_rank_list(r["y_true"]))
                 for _, r in merged.iterrows()]

    return {
        "task_1_pa_ndcg": np.mean(t1_scores),
        "task_2_pa_ndcg": np.mean(t2_scores),
        "mean_pa_ndcg": (np.mean(t1_scores) + np.mean(t2_scores)) / 2
    }

# Ejecutar evaluación si hay ground truth
if "y_true" in df_test.columns:
    print("📊 Evaluando submission...")
    scores = evaluate_submission(DEV_CSV, OUTPUT_SUBMISSION)
    print(json.dumps(scores, indent=2, ensure_ascii=False))

    # Guardar métricas
    metrics_path = Path(OUTPUT_SUBMISSION).with_suffix(".metrics.json")
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump({
            "method_llm": METHOD_LLM,
            "method_multimodal": METHOD_MULTIMODAL,
            "hf_model": HF_LOCAL_MODEL if METHOD_LLM == "hf_local" else HF_API_MODEL,
            "ollama_model": OLLAMA_MODEL if "ollama" in METHOD_LLM else None,
            **scores
        }, f, indent=2, ensure_ascii=False)
    print(f"✅ Métricas guardadas: {metrics_path}")

# %% [markdown]
# ## Archivo .env Recomendado

# %% [markdown]
# ```bash
# # .env file (crear en el mismo directorio)
#
# # Hugging Face Token (opcional pero recomendado para API)
# HF_TOKEN=hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
#
# # Ollama Remote Host (si usas Ollama en la nube)
# OLLAMA_HOST=https://tu-ollama-cloud.com
# OLLAMA_TOKEN=tu-token-opcional
#
# # Otras configuraciones
# PYTHONUNBUFFERED=1
# ```

# %% [markdown]
# ## Guía de Configuración por Plataforma

# %% [markdown]
# ### 1. Hugging Face Local
# ```python
# METHOD_LLM = "hf_local"
# HF_LOCAL_MODEL = "PlanTL-GOB-ES/roberta-base-bne"  # o cualquier modelo español
# ```
# **Requisitos**: GPU recomendada, ~2-8GB VRAM
#
# ### 2. Hugging Face Inference API
# ```python
# METHOD_LLM = "hf_api"
# HF_API_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
# HF_API_TOKEN = "hf_xxx"  # en .env
# ```
# **Requisitos**: Token HF, sin GPU local necesaria
#
# ### 3. Ollama Local
# ```python
# METHOD_LLM = "ollama_local"
# OLLAMA_MODEL = "llama3.1:8b"
# OLLAMA_HOST_LOCAL = "http://localhost:11434"
# ```
# **Requisitos**: Ollama instalado y corriendo localmente
# ```bash
# ollama pull llama3.1:8b
# ollama serve
# ```
#
# ### 4. Ollama Remoto/Cloud
# ```python
# METHOD_LLM = "ollama_remote"
# OLLAMA_MODEL = "llama3.1:8b"
# OLLAMA_HOST_REMOTE = "https://tu-ollama-cloud.com"
# OLLAMA_TOKEN = "xxx"  # en .env
# ```
# **Requisitos**: Servidor Ollama remoto accesible
#
# ### 5. Embeddings (Recomendado para baseline rápido)
# ```python
# METHOD_LLM = "embedding"
# EMBEDDING_MODEL = "sentence-transformers/paraphrasing-multilingual-mpnet-base-v2"
# ```
# **Requisitos**: Mínimos, funciona en CPU
#
# ### 6. NLI (Mejor precisión)
# ```python
# METHOD_LLM = "nli"
# NLI_MODEL = "cross-encoder/nli-deberta-v3-base"
# ```
# **Requisitos**: GPU recomendada

# %% [markdown]
# ## Comparativa de Métodos

# %% [markdown]
# | Método | Velocidad | Precisión | GPU | Coste |
# |--------|-----------|-----------|-----|-------|
# | `embedding` | ⚡⚡⚡ | ⭐⭐⭐ | Opcional | Gratis |
# | `nli` | ⚡⚡ | ⭐⭐⭐⭐ | Recomendada | Gratis |
# | `hf_local` | ⚡ | ⭐⭐⭐⭐ | Requerida | Gratis |
# | `hf_api` | ⚡⚡ | ⭐⭐⭐⭐⭐ | No | Rate limits/Pro |
# | `ollama_local` | ⚡ | ⭐⭐⭐⭐ | Requerida | Gratis |
# | `ollama_remote` | ⚡⚡ | ⭐⭐⭐⭐⭐ | No | Depende del host |

# %% [markdown]
# ## Notas Finales

# %% [markdown]
# - ✅ **Modular**: Cambia `METHOD_LLM` y `METHOD_MULTIMODAL` para experimentar
# - ✅ **Multi-plataforma**: Soporta local y cloud simultáneamente
# - ✅ **Configurable vía .env**: Tokens y hosts externos
# - ✅ **Formato compatible**: Output listo para CodaLab
# - ✅ **Métrica PA-nDCG**: Evaluación integrada si hay ground truth
#
# **¡Éxito en PoliticHeadlinES 2026!** 🎯

# %% [markdown]
# ## Limpieza de Memoria (Opcional)

# %%
import gc
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    print("✓ Memoria GPU liberada")

✓ Device: cuda
✓ Method LLM: ollama_local
✓ Method Multimodal: clip_llm_fusion
✓ Train: 100 | Test: 50 registros
🚀 Ejecutando Task 1 con método: ollama_local
🦙 Conectando a Ollama Local: minimax-m2.7:cloud @ http://localhost:11434
✓ Conexión a Ollama establecida


Ollama Local Scoring:  30%|███       | 3/10 [00:00<00:00,  7.72it/s]

⚠️ Error en Ollama: unauthorized (status code: 401)
⚠️ Error en Ollama: unauthorized (status code: 401)
⚠️ Error en Ollama: unauthorized (status code: 401)



Ollama Local Scoring:  50%|█████     | 5/10 [00:00<00:00,  9.16it/s]

⚠️ Error en Ollama: unauthorized (status code: 401)
⚠️ Error en Ollama: unauthorized (status code: 401)
⚠️ Error en Ollama: unauthorized (status code: 401)


...

⚠️ Error en Ollama: unauthorized (status code: 401)
⚠️ Error en Ollama: <!doctype html><meta charset="utf-8"><meta name=viewport content="width=device-width, initial-scale=1"><title>429</title>429 Too Many Requests (status code: 429)



Task 2: 100%|██████████| 50/50 [00:51<00:00,  1.03s/it]


✅ Submission guardado: results_llm.csv


,id,task_1,task_2
0,18bd3b59c79aaf46ca09344dff4a97a8a9722b3955bd00...,t1 t2 t3 t4 t5 t6 t7 t8 t9 t10,t1 t3 t2 t5 t4 t6 t7 t8 t10 t9
1,5650055bae475cf2dde50d805b6d57e50dd19591c188fb...,t1 t2 t3 t4 t5 t6 t7 t8 t9 t10,t1 t3 t2 t4 t5 t6 t7 t8 t9 t10
2,384683aeb9c766e80e8df44fd47c7ced26ee8c4083fdce...,t1 t2 t3 t4 t5 t6 t7 t8 t9 t10,t1 t2 t3 t4 t5 t6 t8 t7 t9 t10
3,5080ebec5612904a3a3a03f195ab24bd13ed4e2bd814f4...,t1 t2 t3 t4 t5 t6 t7 t8 t9 t10,t1 t3 t2 t4 t5 t6 t7 t8 t9 t10
4,f1acf2f90e0fdec79a711a9782ef6511ea8eb5972291bb...,t1 t2 t3 t4 t5 t6 t7 t8 t9 t10,t1 t3 t2 t4 t5 t6 t7 t8 t9 t10


📊 Evaluando submission...
{
  "task_1_pa_ndcg": 0.1371031537811689,
  "task_2_pa_ndcg": 0.15700609329453272,
  "mean_pa_ndcg": 0.14705462353785081
}
✅ Métricas guardadas: results_llm.metrics.json
✓ Memoria GPU liberada


In [ ]:
METHOD_LLM = "hf_api"
HF_API_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
HF_API_TOKEN = os.getenv("HF_TOKEN")  # desde .env

In [ ]:
METHOD_LLM = "ollama_remote"
OLLAMA_HOST_REMOTE = os.getenv("OLLAMA_HOST", "https://tu-cloud.com")
OLLAMA_TOKEN = os.getenv("OLLAMA_TOKEN")